# Polyomino decision classifier on one v5e TPU

Fine-tune the pinned LFM2.5 Base backbone with an 8-output action head. The published decision split supplies raw 11-column game states and one-hot expert actions. Prompts are derived as rows are read, and each row is visited once in seeded order. Checkpoints preserve the update cursor.

This notebook keeps downloads and a 2-update smoke under ephemeral `/content/polyomino-smoke`. The optional long run requires a mounted persistent checkpoint directory. Run the cells in order. The notebook kernel never owns the TPU; the Python 3.12 child process does.


In [ ]:
from pathlib import Path
import subprocess
import sys

CHECKOUT = Path("/content/minifield-training")
ROOT = Path("/content/polyomino-smoke")
VENV = ROOT / "venv"
PYTHON = VENV / "bin/python"
SOURCE_REVISION = "b79de2f08863b63f0f6063b8c9f14a645eab61c9"
BASE_REVISION = "9d2be5519834990d30996f878b6771cccbd24f2c"
assert Path("/usr/bin/python3.12").is_file()
ROOT.mkdir(parents=True, exist_ok=True)

def run_child(*command: str, cwd: Path | None = None) -> str:
    with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        assert process.stdout is not None
        lines = []
        for line in process.stdout:
            print(line, end="", flush=True)
            lines.append(line)
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    return "".join(lines)

UV = ROOT / "tooling/bin/uv"
if not UV.is_file():
    run_child(sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
              "--target", str(ROOT / "tooling"), "uv==0.11.30")
assert run_child(str(UV), "--version").strip() == "uv 0.11.30"


In [ ]:
if not (CHECKOUT / ".git").is_dir():
    assert not CHECKOUT.exists(), f"Expected a clean path: {CHECKOUT}"
    run_child("git", "clone", "https://github.com/Minifield-Labs/minifield-training.git", str(CHECKOUT))
assert not run_child("git", "status", "--porcelain", cwd=CHECKOUT).strip()
run_child("git", "fetch", "origin", SOURCE_REVISION, cwd=CHECKOUT)
run_child("git", "checkout", "--detach", SOURCE_REVISION, cwd=CHECKOUT)
assert run_child("git", "rev-parse", "HEAD", cwd=CHECKOUT).strip() == SOURCE_REVISION
assert (CHECKOUT / "examples/polyomino/train.py").is_file()


In [ ]:
if not PYTHON.is_file():
    run_child(str(UV), "venv", "--python", "/usr/bin/python3.12", str(VENV))
run_child(str(UV), "pip", "install", "--python", str(PYTHON),
          "jax[tpu]==0.7.2", "tensorflow-cpu==2.20.0")
run_child(str(UV), "pip", "install", "--python", str(PYTHON),
          ".[numerical,storage,text,hub]", cwd=CHECKOUT)
probe = "import jax; devices=jax.devices(); assert jax.__version__ == '0.7.2' and len(devices) == 1 and devices[0].platform == 'tpu', devices; print(devices)"
run_child(str(PYTHON), "-c", probe)


In [ ]:
MODEL_DIR = ROOT / "base-model"
DATASET_CACHE = ROOT / "dataset-cache"
download = "from huggingface_hub import snapshot_download; import sys; snapshot_download(repo_id='LiquidAI/LFM2.5-230M-Base', revision=sys.argv[1], allow_patterns=['config.json', 'tokenizer.json', 'model.safetensors'], local_dir=sys.argv[2])"
run_child(str(PYTHON), "-c", download, BASE_REVISION, str(MODEL_DIR), cwd=CHECKOUT)
assert all((MODEL_DIR / name).is_file() for name in ("config.json", "tokenizer.json", "model.safetensors"))
print("Base weights ready. The dataset downloads on the first training call.")


The first call downloads and prepares the pinned 5-file Parquet split. That one-time stage can take several minutes and needs space for the 2.5 GB download plus the Arrow cache. The smoke then executes 2 updates and writes a full checkpoint. Read its replay to see a game played by the learned head.


In [ ]:
CHECKPOINTS = ROOT / "checkpoints"
common = ["--model-dir", str(MODEL_DIR), "--dataset-cache", str(DATASET_CACHE),
          "--checkpoint-root", str(CHECKPOINTS), "--run-id", "polyomino-v5e-smoke-1",
          "--platform", "tpu", "--checkpoint-every", "2", "--report-every", "1"]
run_child(str(PYTHON), "-u", "-m", "examples.polyomino.train", *common,
          "--resume-latest", "--skip-if-resumed", "--max-steps", "2",
          "--eval-games", "1", "--eval-max-ticks", "1000", cwd=CHECKOUT)
assert (CHECKPOINTS / "step-00000002" / "manifest.json").is_file()


The long run uses the same frozen decision split. Set `PERSISTENT_ROOT` to a mounted, writable directory before starting. A resumed call picks up at the next decision update; the dataset order never restarts. Run gameplay evaluation after training so its board stepping and replay writing don't interrupt the TPU updates.


In [ ]:
PERSISTENT_ROOT: Path | None = None  # Set to an existing mounted absolute path.
if PERSISTENT_ROOT is None:
    print("Set PERSISTENT_ROOT to run training.")
else:
    assert PERSISTENT_ROOT.is_absolute() and PERSISTENT_ROOT.is_dir()
    long_checkpoints = PERSISTENT_ROOT / "polyomino-checkpoints"
    run_id = "polyomino-base-decisions-v1"
    run_child(str(PYTHON), "-u", "-m", "examples.polyomino.train",
              "--model-dir", str(MODEL_DIR), "--dataset-cache", str(DATASET_CACHE),
              "--checkpoint-root", str(long_checkpoints), "--run-id", run_id,
              "--platform", "tpu", "--resume-latest", "--max-hours", "3",
              "--checkpoint-every", "5000", "--report-every", "50",
              "--eval-games", "0", cwd=CHECKOUT)
    checkpoints = sorted(long_checkpoints.glob("step-*/manifest.json"))
    assert checkpoints, "Training produced no checkpoint"
    latest = checkpoints[-1].parent
    run_child(str(PYTHON), "-u", "-m", "examples.polyomino.evaluate",
              "--model-dir", str(MODEL_DIR), "--checkpoint", str(latest),
              "--run-id", run_id, "--replay-dir", str(long_checkpoints / "replays"),
              "--games", "3", "--max-ticks", "2000", cwd=CHECKOUT)
